# 9 · Surapprentissage et sous-apprentissage — solutions (notebook exécuté)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/ch09_overfitting/05_solutions.ipynb)

Les réponses des exercices, exécutées. Les démarches détaillées (le *pourquoi*, les erreurs fréquentes, les variantes) sont dans `05_solutions.md`. Les cellules marquées `answer` enregistrent les réponses vérifiées par `wb.check` (`tools/build_answers.py`).

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | 9.Q1 à 9.Q10, 9.R2, 9.R3, 9.1 à 9.7, 9.9 | vérifier tes réponses courtes | 🧠 🔁 ✏️ ∂ 📈 | ★ à ★★★ | |
| A | 9.12 à 9.17 | sous- et surapprentissage, mesures d'erreur, features polynomiales, moindres carrés et Ridge dans mylearn.linear (*prochaine session de génération*) | | | |
| B | 9.18 à 9.23 | courbes de validation, chemins de régularisation, early stopping, courbes d'apprentissage, Ridge contre Lasso, Lasso par descente de coordonnées (*prochaine session de génération*) | | | |
| C | 9.24 à 9.27 | biais et variance mesurés par simulation, droites bayésiennes sur une grille (*prochaine session de génération*) | | | |
| D | 9.28 à 9.31 | bugs de régularisation, refactorisation testée, double descente, défi California (*prochaine session de génération*) | | | |

In [1]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        if pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("ref")  # reference implementation

🧪 Deep Learning Workbook : environnement
   Plateforme : Local (Linux x86_64, 2 CPU)
   Python 3.13.13 | numpy 2.1.3 | pandas 2.2.3 | matplotlib 3.10.0 | scikit-learn 1.6.1 | scipy 1.16.3 | torch 2.11.0+cpu | torchvision 0.26.0+cpu
   Device : cpu | Graine : 42 | FAST_MODE : True


## Objectifs et rappel express

- Diagnostiquer l'underfitting et l'overfitting sur des courbes d'entraînement, de validation et d'apprentissage.
- Écrire et vérifier contre scikit-learn les mesures d'erreur d'une régression, les features polynomiales, les moindres carrés, Ridge et Lasso.
- Appliquer l'early stopping avec patience, et choisir la régularisation par validation croisée.
- Mesurer le biais² et la variance d'une famille de modèles, et ajuster une droite à la manière bayésienne.

**Rappel express.** $\mathrm{MSE} = \frac{1}{n}\sum_i (y_i - \hat{y}_i)^2$, $\mathrm{MAE} = \frac{1}{n}\sum_i |y_i - \hat{y}_i|$, $R^2 = 1 - SS_{\text{res}}/SS_{\text{tot}}$ ; moindres carrés : $a = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sum (x_i - \bar{x})^2}$, $b = \bar{y} - a\bar{x}$ ; Ridge : $(\mathbf{X}_c^\top\mathbf{X}_c + \alpha\mathbf{I})\mathbf{w} = \mathbf{X}_c^\top\mathbf{y}_c$ ; Lasso : $w_j = S(\rho_j, \alpha)/z_j$ avec $S(z, \gamma) = \operatorname{signe}(z)\max(|z| - \gamma, 0)$ ; erreur attendue = biais² + variance + $\sigma^2$. En Python : `np.linalg.lstsq`, `np.linalg.solve`, `sklearn.preprocessing.PolynomialFeatures`, `sklearn.linear_model.LinearRegression`, `Ridge`, `Lasso`, `sklearn.model_selection.learning_curve`, `validation_curve`.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ 9.1, 9.4, 9.5 et 9.7, ∂ 9.2, 9.3 et 9.6, 📈 9.9)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"E"`) ; plusieurs choix : les lettres collées (`"BD"`) ; plusieurs nombres : une liste (`[4, 7]`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », le quiz Q11, le rappel R1, la réflexion et l'entretien se corrigent avec `05_solutions.md`.

In [2]:
# The short answers only use the numbers of their statements (02_exercices.md)
import math

import numpy as np

# R2: probabilities given to the right class of four examples
P_R2 = np.array([0.9, 0.6, 0.25, 0.8])
CE_R2 = float(-np.mean(np.log(P_R2)))                          # nats

# R3: four measurements
S_R3 = np.array([3.0, 5.0, 6.0, 10.0])

# 9.1: five targets and five predictions
Y_91 = np.array([3.0, 5.0, 4.0, 8.0, 10.0])
YHAT_91 = np.array([2.0, 5.5, 5.0, 7.5, 9.0])
R_91 = Y_91 - YHAT_91
SST_91 = float(np.sum((Y_91 - Y_91.mean()) ** 2))
Y2_91 = Y_91.copy()
Y2_91[4] = 20.0                                                # the 5th target becomes 20
R2_91 = Y2_91 - YHAT_91

# 9.2: five points and their least squares line
X_92 = np.array([0.0, 1.0, 2.0, 3.0, 4.0])
Y_92 = np.array([1.0, 3.0, 2.0, 5.0, 7.0])
A_92 = float(np.sum((X_92 - X_92.mean()) * (Y_92 - Y_92.mean())) / np.sum((X_92 - X_92.mean()) ** 2))
B_92 = float(Y_92.mean() - A_92 * X_92.mean())
RES_92 = Y_92 - (A_92 * X_92 + B_92)

# 9.3 and 9.6: one feature, no intercept
X_93 = np.array([-2.0, -1.0, 0.0, 1.0, 2.0])
Y_93 = np.array([-3.0, -1.0, 0.0, 2.0, 2.0])
SXY_93, SXX_93 = float(X_93 @ Y_93), float(X_93 @ X_93)


def ridge_1d(lam):
    """The weight of the 1-D ridge regression without intercept (∂ 9.3)."""
    return SXY_93 / (SXX_93 + lam)


def soft_threshold(z, gamma):
    """S(z, gamma) = sign(z) max(|z| - gamma, 0)."""
    return math.copysign(max(abs(z) - gamma, 0.0), z) + 0.0


N_96 = len(X_93)
RHO_96, Z_96 = SXY_93 / N_96, SXX_93 / N_96

# 9.4: four models at three points, and the ideal curve
P_94 = np.array([[1.5, 2.0, 2.0], [0.5, 3.0, 2.5], [1.0, 2.5, 1.5], [1.0, 2.5, 2.0]])
F_94 = np.array([1.0, 2.0, 3.0])
AVG_94 = P_94.mean(axis=0)
BIAS2_94 = float(np.mean((AVG_94 - F_94) ** 2))
VAR_94 = float(np.mean(P_94.var(axis=0)))                       # ddof=0: divide by the 4 models

# 9.5: the validation losses of 15 epochs
VAL_95 = [0.90, 0.72, 0.61, 0.55, 0.53, 0.54, 0.52, 0.53, 0.55, 0.54, 0.56, 0.57, 0.58, 0.60, 0.61]


def early_stopping(losses, patience, min_delta=0.0):
    """(epoch of the stop, epoch whose weights are kept), epochs numbered from 1 (the rule of the fiche)."""
    best, kept, wait = math.inf, None, 0
    for epoch, loss in enumerate(losses, start=1):
        if loss < best - min_delta - 1e-12:                    # a strict improvement (1e-12: float rounding)
            best, kept, wait = loss, epoch, 0
        else:
            wait += 1
            if wait == patience:
                return epoch, kept
    return len(losses), kept


# 9.7: nine lines (a, b), a prior with weights 4, 2 and 1, a coarse likelihood of the gap r
LINES_97 = [(a, b) for a in (-1, 0, 1) for b in (-1, 0, 1)]
PRIOR_97 = {line: {0: 4, 1: 2, 2: 1}[abs(line[0]) + abs(line[1])] / 16 for line in LINES_97}


def likelihood_97(line, point):
    gap = abs(point[1] - (line[0] * point[0] + line[1]))
    return {0: 1.0, 1: 0.5, 2: 0.1}.get(gap, 0.0)


def normalised(weights):
    total = sum(weights.values())
    return {line: weight / total for line, weight in weights.items()}


POST1_97 = normalised({line: PRIOR_97[line] * likelihood_97(line, (1, 1)) for line in LINES_97})
POST2_97 = normalised({line: POST1_97[line] * likelihood_97(line, (-1, 0)) for line in LINES_97})

**9.Q1 — Sur- ou sous-apprentissage ? Définitions et symptômes**

In [3]:
wb.record("9.Q1a", "B", mistakes={"18 % sur la validation contre 0,5 % sur l'entraînement : regarde l'écart": "A",
          "l'erreur d'entraînement est très basse : le modèle réussit ses propres exemples": "C"})
wb.record("9.Q1b", "C", mistakes={"les deux erreurs sont proches, mais compare leur niveau à celui d'un humain": "A",
          "l'écart est petit : ce n'est pas le symptôme de l'overfitting": "B"})
wb.record("9.Q1c", "A", mistakes={"un écart d'un point n'est pas le symptôme de l'overfitting": "B",
          "3 % contre 2 % pour un humain, et un seul point d'écart avec la validation : est-ce le portrait d'un modèle qui n'a pas appris ?": "C"})
wb.record("9.Q1d", False, mistakes={"cherche, parmi les trois modèles, un contre-exemple à cette phrase": True})

WB_ANSWER {"hash": "f5cd94f63650588ae24ed82037b88730fc3aa15822fa26bde0258efd7033bd13", "id": "9.Q1a", "kind": "str", "mistakes": {"8d08051b54d6fe7eabaa5004d8131a6ea2bd5fcbeca3f88110353588b17b9aa8": "18 % sur la validation contre 0,5 % sur l'entraînement : regarde l'écart", "9ea17edab9fc8c86cfa6fb6bf24529617676267b8752004deb7e260d9303ad0b": "l'erreur d'entraînement est très basse : le modèle réussit ses propres exemples"}}
📝 Ex 9.Q1a : réponse enregistrée (str).
WB_ANSWER {"hash": "15acae4c61c1172a5dd745ef142724470ff38054efa78372584c44bfae968c1f", "id": "9.Q1b", "kind": "str", "mistakes": {"9fad303251e446f18707020ff7e61090e0dd7d70f3f35cec90666589c1ca0506": "les deux erreurs sont proches, mais compare leur niveau à celui d'un humain", "bee012e210e417633d65f82100ffdc9c07c7983069b12f745ed578ad40556daf": "l'écart est petit : ce n'est pas le symptôme de l'overfitting"}}
📝 Ex 9.Q1b : réponse enregistrée (str).
WB_ANSWER {"hash": "050c902d9e3c5b7c2ba33c661bc87ecd6d9ebd43f9f107e29ff31d5f1b64b0f

**9.Q2 — Walter et sa moustache : qu'est-ce qui a été mal appris ?**

In [4]:
wb.record("9.Q2a", "C", mistakes={"les prénoms ont bien été retenus : on les a même ressortis": "A",
          "c'est le lien entre l'apparence et le prénom qui a trompé, pas la mémoire de l'apparence": "B",
          "l'histoire montre une règle qui trompe, pas un oubli": "D"})
wb.record("9.Q2b", "B", mistakes={"pendant le mariage, chaque invité déjà rencontré était reconnu": "A"})
wb.record("9.Q2c", True, mistakes={"relis le §9.5 du livre : ce qu'on aurait pu remarquer d'autre chez Walter": False})
wb.record("9.Q2d", "D", mistakes={"le learning rate règle la taille des pas, pas ce que le modèle remarque": "A",
          "un test plus grand ne changerait pas ce qui a été appris": "B",
          "le problème vient de ce qui permet de reconnaître les exemples, pas de la loss": "C"})

WB_ANSWER {"hash": "de649ae36fb2b3719bc4e3337c48720108184702162008fe53a101f431c7773a", "id": "9.Q2a", "kind": "str", "mistakes": {"04168d90ffbea3501bfdf47c7ee40ecfffcf6e00a2934354c3d7252d6c2b4a93": "les prénoms ont bien été retenus : on les a même ressortis", "81371f702d5649995f228a84b3afbf5cc41e54aa48817ca81140e53e3f188897": "l'histoire montre une règle qui trompe, pas un oubli", "d3b998dc42cc53a73297969abe0ad7a26e5315006bcc1068769ca572e078cb0a": "c'est le lien entre l'apparence et le prénom qui a trompé, pas la mémoire de l'apparence"}}
📝 Ex 9.Q2a : réponse enregistrée (str).
WB_ANSWER {"hash": "202c50f0914c7261dc1320bd61b5c18a97757c0f82af6ecd48e324a4c1ad8f46", "id": "9.Q2b", "kind": "str", "mistakes": {"f990dd74fb804a9dd7b5bce2e6b56547e5d2d664caff3fc71b360bf3c3f9160b": "pendant le mariage, chaque invité déjà rencontré était reconnu"}}
📝 Ex 9.Q2b : réponse enregistrée (str).
WB_ANSWER {"hash": "37be688c5945951dbe2bcca2f73e0101be0a516b2eefe29b135a3e5ca965ae69", "id": "9.Q2c", "kind": 

**9.Q3 — Sous-apprentissage : les vrais remèdes**

In [5]:
wb.record("9.Q3a", "D", mistakes={"une fuite donnerait un score trop beau, pas un score bas": "A",
          "l'écart entre entraînement et validation est minuscule": "B",
          "un modèle plus souple fait bien mieux sur la même validation": "C"})
wb.record("9.Q3b", "ACE", mistakes={"plus d'exemples du même genre rend-il une régression linéaire plus souple ? (fiche §9.2.2)": "ABCE",
          "augmenter la régularisation rend le modèle encore plus rigide": "ACDE",
          "une pénalité plus faible donne-t-elle plus ou moins de liberté au modèle ?": "AE",
          "relis le diagnostic de a) : le problème est-il la variance ?": "B",
          "un modèle plus souple atteint 0,80 sur la même validation : ce remède-là ne serait-il pas utile ?": "AC",
          "ajouter des puissances et des produits de features, c'est aussi rendre le modèle plus souple": "CE"})
wb.record("9.Q3c", False, mistakes={"les deux R² sont déjà presque égaux : sur quoi plus d'exemples agiraient-ils ?": True})

WB_ANSWER {"hash": "6949dbcb296e249bddb7308ed7c7de602f524475d2d4e6709aae41ff6c3300f8", "id": "9.Q3a", "kind": "str", "mistakes": {"033871b01bf140f36b8bb306a6e9525d4bd0d9938ee437ab8ec166812a13eed5": "un modèle plus souple fait bien mieux sur la même validation", "4a9937de369835e7fdc16ebd98d77375f823c99d45622e8931364bca52eeac5b": "une fuite donnerait un score trop beau, pas un score bas", "bfb7335ae8ca4ba2482c4fa0c0c639453881092fcff50bbc5bec10fad7642769": "l'écart entre entraînement et validation est minuscule"}}
📝 Ex 9.Q3a : réponse enregistrée (str).
WB_ANSWER {"hash": "c73ac5519a1212e635de1c9b6b469bf6d9fbc35ebaa34a06d8a33ca915c345b8", "id": "9.Q3b", "kind": "str", "mistakes": {"01ba5b01378bf013685abb2020163e3cf00f6fe35e518f349b140d42c788c27b": "une pénalité plus faible donne-t-elle plus ou moins de liberté au modèle ?", "50a59068b9ad43e181d139e68d0fea9bc71f3f36da4b96a513a08bc9d982207c": "plus d'exemples du même genre rend-il une régression linéaire plus souple ? (fiche §9.2.2)", "8944

**9.Q4 — Courbes d'erreur : où commence le surapprentissage ?**

In [6]:
wb.record("9.Q4a", 20, mistakes={"c'est l'epoch de la plus basse erreur d'entraînement : la question porte sur la validation": 40,
          "lis toute la ligne de validation : la question demande sa plus petite valeur, pas le moment où elle se met à remonter": 25})
wb.record("9.Q4b", True, mistakes={"regarde la ligne « entraînement » au-delà de cette epoch": False})
wb.record("9.Q4c", "A", mistakes={"à l'epoch 35, l'erreur d'entraînement est basse et l'écart avec la validation est grand": "B",
          "la validation suffit pour ce diagnostic ; le test est gardé pour l'évaluation finale": "C"})
wb.record("9.Q4d", False, mistakes={"l'erreur de validation est mesurée sur un échantillon de données nouvelles : relis l'encadré ⚠️ de la fiche §9.3": True})

WB_ANSWER {"hash": "3a35a850ded68fbbefd5d55605e0bd548ac1b921441214ace22482398d2d0007", "id": "9.Q4a", "kind": "int", "magnitude": 1, "mistakes": {"a361cdb279131c72f0e2503144d635f45eb777ec27cc9024b82a2c7fdc42f9ed": "lis toute la ligne de validation : la question demande sa plus petite valeur, pas le moment où elle se met à remonter", "f0b380a25f62accd7bd379029dbab453284bbc70bc097e7de1fdddc852e77792": "c'est l'epoch de la plus basse erreur d'entraînement : la question porte sur la validation"}, "sign": 1}
📝 Ex 9.Q4a : réponse enregistrée (int).
WB_ANSWER {"hash": "8825fa7b60aabe66b7eecde70785bf61316013508c11ba1de9b0f03a05e93458", "id": "9.Q4b", "kind": "bool", "mistakes": {"6101105a0e06f88913be7d05d9159d543e96140faaa43872e43ef1fa6f836a43": "regarde la ligne « entraînement » au-delà de cette epoch"}}
📝 Ex 9.Q4b : réponse enregistrée (bool).
WB_ANSWER {"hash": "681e4e9afce24f0da254b0269f817591ea13dd766d8b8688d8f48a4801c70c5f", "id": "9.Q4c", "kind": "str", "mistakes": {"7a33b685ac8b5a95c24

**9.Q5 — Un point isolé : frontière tordue ou frontière simple ?**

In [7]:
wb.record("9.Q5a", "D", mistakes={"un centroïde est le centre d'un groupe de points": "A",
          "une fuite fait passer une information du test vers le modèle": "B",
          "ce point est remarquable par sa position au milieu de l'autre classe (les vecteurs de support viendront avec les SVM, ch. 13)": "C"})
wb.record("9.Q5b", True, mistakes={"compte les points d'entraînement mal classés par chacune des deux frontières": False})
wb.record("9.Q5c", "A", mistakes={"regarde de quel côté de la frontière simple tombe tout ce voisinage": "B"})
wb.record("9.Q5d", False, mistakes={"un point aberrant peut être une vraie valeur, rare : il faut d'abord savoir d'où il vient": True})

WB_ANSWER {"hash": "b8070469e0eaa969325ab02bde2c868cbd50c91d0385d79276d1271d5c823f6d", "id": "9.Q5a", "kind": "str", "mistakes": {"00e4655477632da82e813285d93762e2e612c046d33d7eee9f1393170ed9fb55": "ce point est remarquable par sa position au milieu de l'autre classe (les vecteurs de support viendront avec les SVM, ch. 13)", "11d421d9e522b0bedcdc19ebf45f7439c07965d9bf65f42376f088a3487d7f78": "un centroïde est le centre d'un groupe de points", "8d29a8fbd2fae2aa98ebcbbb907350ed9946f5876e6b7e104260f31f2cda36ed": "une fuite fait passer une information du test vers le modèle"}}
📝 Ex 9.Q5a : réponse enregistrée (str).
WB_ANSWER {"hash": "95c75aa3316513d59ff277dced8b241f4796fe71e37d503a253a70786826fa1b", "id": "9.Q5b", "kind": "bool", "mistakes": {"d5cb4fef57131a52393e33e76b47341fe391ac189190869912900fdad130f88a": "compte les points d'entraînement mal classés par chacune des deux frontières"}}
📝 Ex 9.Q5b : réponse enregistrée (bool).
WB_ANSWER {"hash": "1742ddc13ab13da47f75142c72216adc2298ef0

**9.Q6 — Early stopping : quand s'arrêter, et pourquoi c'est délicat**

In [8]:
wb.record("9.Q6a", "D", mistakes={"décider sur le test, c'est déjà s'en servir pour régler le modèle (ch. 8)": "A",
          "l'erreur d'entraînement baisse presque toujours : elle ne signale pas l'overfitting": "B",
          "le learning rate est un réglage, pas une mesure de la généralisation": "C"})
wb.record("9.Q6b", "B", mistakes={"l'erreur d'entraînement continue en général de baisser": "A",
          "le test n'est pas consulté pendant l'entraînement": "C",
          "28 epochs est la valeur d'une figure du livre, pas une règle": "D"})
wb.record("9.Q6c", 5, fractional="un nombre d'epochs est entier",
          mistakes={"compte les epochs sans amélioration nécessaires pour que l'attente atteigne la patience": 4,
          "l'epoch de la meilleure loss ne compte pas dans l'attente": 6})
wb.record("9.Q6d", "A", mistakes={"la dernière epoch suit plusieurs epochs sans amélioration : ses poids ne sont pas les meilleurs": "B",
          "moyenner des poids est une autre technique, qui n'est pas l'early stopping": "C",
          "la première epoch n'a presque rien appris": "D"})

WB_ANSWER {"hash": "3952f40a6cc12e374975fc8adae0a086f848b146b0452dafcacb9a955b820212", "id": "9.Q6a", "kind": "str", "mistakes": {"32a14543e8bad7533ff50bd1aae3ab1e3a8f51842708f55693935b3d0a638e60": "l'erreur d'entraînement baisse presque toujours : elle ne signale pas l'overfitting", "c5b597632a7d4fb52baae0b4499ab6883b75169c8738d7bd7ce7b063c84136b5": "décider sur le test, c'est déjà s'en servir pour régler le modèle (ch. 8)", "d6f3de849d890298c2dc70317a817d375654b6b6268fe71a2f44378d5fb9f738": "le learning rate est un réglage, pas une mesure de la généralisation"}}
📝 Ex 9.Q6a : réponse enregistrée (str).
WB_ANSWER {"hash": "990fa93a4183142876a140477a00f0d159ce96c84e2646b558d6dcad02b7945a", "id": "9.Q6b", "kind": "str", "mistakes": {"55a444d3210a163d3ec9457ea2e6c4ab8c1e86d74b371e7969e05ec8262feed0": "28 epochs est la valeur d'une figure du livre, pas une règle", "8a0aaffa6c5d739dc5d3864991b15d4f76fcb672c09ffdf793751079928babf4": "l'erreur d'entraînement continue en général de baisser", "

**9.Q7 — Régularisation : ce que change λ**

In [9]:
wb.record("9.Q7a", "C", mistakes={"la pénalité grandit avec la taille des poids : que fait la minimisation quand elle pèse plus lourd ?": "A",
          "la pénalité dépend des poids : elle les change": "B",
          "la pénalité ne rend pas les poids égaux": "D"})
wb.record("9.Q7b", "A", mistakes={"la loss sur les données est la plus basse sans pénalité : que se passe-t-il quand la solution s'en éloigne ?": "B"})
wb.record("9.Q7c", False, mistakes={"si l'on minimisait la loss d'entraînement en λ, quelle valeur obtiendrait-on toujours ?": True})
wb.record("9.Q7d", "A", mistakes={"C joue le rôle de l'inverse de λ (encadré 💼 de la fiche §9.5)": "B"})
wb.record("9.Q7e", True, mistakes={"avec des poids écrasés à 0, que reste-t-il de la prédiction, l'ordonnée à l'origine n'étant pas pénalisée ?": False})

WB_ANSWER {"hash": "3521d71eb7f9d7f0f3783029550be01b78ad58c42639cd2b114dc7c1a4bc3825", "id": "9.Q7a", "kind": "str", "mistakes": {"48349e78131ebc2ffb1fa983b3feb9848a4d922e10928c208cfc1c71b13fcd4f": "la pénalité dépend des poids : elle les change", "58e7d4a6a12ccd2f4e7d488801511a14afea6a19cdcf4aef3feaa52444738ec4": "la pénalité grandit avec la taille des poids : que fait la minimisation quand elle pèse plus lourd ?", "74df0918f512960d6a203f4b7b7b4ae42f2fbe6d8b161649b7640ec76fc1c80c": "la pénalité ne rend pas les poids égaux"}}
📝 Ex 9.Q7a : réponse enregistrée (str).
WB_ANSWER {"hash": "d4cae3a85d33b9c4ec5ca053ffead0c6982149f3a587bbb21014a9ac6ee25869", "id": "9.Q7b", "kind": "str", "mistakes": {"94eebe9b413b0ed90060430733f7fc2d1effa1f5f79a273b1ab8ee0b6699f1b2": "la loss sur les données est la plus basse sans pénalité : que se passe-t-il quand la solution s'en éloigne ?"}}
📝 Ex 9.Q7b : réponse enregistrée (str).
WB_ANSWER {"hash": "c01f382261faf46236409fd95986ccb0e22cfbddbc786994dd86e0c87

**9.Q8 — Pénalité sur les poids, dropout, batchnorm : même objectif ?**

In [10]:
wb.record("9.Q8a", "ABCE", mistakes={"l'early stopping est aussi une régularisation : il empêche d'apprendre le détail": "ABE",
          "l'augmentation de données vise aussi à mieux généraliser": "ABC",
          "un learning rate plus grand sert d'abord à aller plus vite": "ABCDE",
          "le dropout est une régularisation propre aux réseaux": "ACE",
          "une pénalité L2 sur les poids est le premier exemple de régularisation de la fiche §9.5": "BCE"})
wb.record("9.Q8b", True, mistakes={"relis l'encadré ⚠️ de la fiche §9.5 sur l'intention des auteurs de la batchnorm": False})
wb.record("9.Q8c", "C", mistakes={"le dropout agit sur les neurones, pas sur les exemples": "A",
          "le dropout est temporaire : à chaque pas, d'autres neurones sont éteints": "B",
          "aucun neurone n'est supprimé pour de bon": "D"})
wb.record("9.Q8d", False, mistakes={"compare les deux panneaux de la figure l1_l2.png : laquelle des deux pénalités met des poids exactement à zéro ?": True})

WB_ANSWER {"hash": "682e56268f617eea140dc461b14d157731efe46cf36ddf48592bbef67881801d", "id": "9.Q8a", "kind": "str", "mistakes": {"49bba5e0dfb13f3300584a84ea24561459127a71541256cdc7df5ba6282dbd1e": "l'augmentation de données vise aussi à mieux généraliser", "5019d1d42dd4861bdbcd79b973e573d4df6d0ebbf8ff424c1bd6d1521227705f": "l'early stopping est aussi une régularisation : il empêche d'apprendre le détail", "5d797a8852f4b9ab9da099cb09885638227682ccb1639fbfb42b4c4ec6bc6f00": "une pénalité L2 sur les poids est le premier exemple de régularisation de la fiche §9.5", "94d9439893e0374b9b8097c4bbda359d9de3894eaa65870832795ced96f74351": "le dropout est une régularisation propre aux réseaux", "9d1e1410ec3ab695dbe025ebbe016245b00c5eca74396a9b8f51ad02ecfb05f3": "un learning rate plus grand sert d'abord à aller plus vite"}}
📝 Ex 9.Q8a : réponse enregistrée (str).
WB_ANSWER {"hash": "b8520f927bdaf70bd6f499bf234675270c0bbad73e2fa40bc72f0bc5e674d7d7", "id": "9.Q8b", "kind": "bool", "mistakes": {"898e

**9.Q9 — Biais et variance : des propriétés d'une famille de courbes**

In [11]:
wb.record("9.Q9a", False, mistakes={"il faut une moyenne et une dispersion : sur combien de courbes les calculer ?": True})
wb.record("9.Q9b", "A", mistakes={"comparer aux points bruités, c'est mesurer l'erreur d'entraînement": "B",
          "le test sert à l'évaluation, pas à définir le biais": "C",
          "la courbe moyenne est la référence de la variance": "D"})
wb.record("9.Q9c", "D", mistakes={"c'est la définition du biais": "A",
          "c'est le troisième terme de la décomposition": "B",
          "l'erreur d'entraînement ne mesure pas la dispersion entre modèles": "C"})
wb.record("9.Q9d", "B", mistakes={"même un modèle parfait se trompe sur des mesures bruitées": "A",
          "le biais est nul par hypothèse": "C",
          "l'erreur dépend de la taille du bruit, pas d'une constante": "D"})

WB_ANSWER {"hash": "51b5b15674549ee9a9d5fac0b8e2481eb1936f18f75bdf70d7cd888e853de83e", "id": "9.Q9a", "kind": "bool", "mistakes": {"14ef982ee3095e714e11cbc660e88f2e039a980924df0c6a877819421ad5a035": "il faut une moyenne et une dispersion : sur combien de courbes les calculer ?"}}
📝 Ex 9.Q9a : réponse enregistrée (bool).
WB_ANSWER {"hash": "5c1580fbee12c7edc39412f6c9a32a46552f1fb8cee07b5038162257ef293b58", "id": "9.Q9b", "kind": "str", "mistakes": {"3fcfa5ccc907e9a2b9904670ab1c1c2ef0900de6af105a461e6664d87f7052ac": "le test sert à l'évaluation, pas à définir le biais", "abb9c158b7392ef245a26879d6f3328d138b041044ecfedcc22ef106cd3a4ebd": "comparer aux points bruités, c'est mesurer l'erreur d'entraînement", "c10ce5568fde86a23a951dd2a9df67e3f04f93f30ab782b96b91ddb4e2e2073d": "la courbe moyenne est la référence de la variance"}}
📝 Ex 9.Q9b : réponse enregistrée (str).
WB_ANSWER {"hash": "905c78dde2f8ef29d8a7559328418ae90766aa45b52b34c20f3040309b5647c7", "id": "9.Q9c", "kind": "str", "mistake

**9.Q10 — Courbes raides ou souples : qui a quel biais, quelle variance ?**

In [12]:
wb.record("9.Q10a", "A", mistakes={"un polynôme de degré 15 peut suivre une courbe ondulée, en moyenne sur les jeux": "B"})
wb.record("9.Q10b", "B", mistakes={"des droites ajustées à des jeux de 30 points se ressemblent beaucoup": "A"})
wb.record("9.Q10c", True, mistakes={"où les points qui retiennent le polynôme sont-ils les plus rares ?": False})
wb.record("9.Q10d", True, mistakes={"avec plus de points par jeu, chaque polynôme dépend-il autant des points tirés ?": False})

WB_ANSWER {"hash": "851fa4610f9444973f2071360820f24bf160cda5276b1c9c8c1e1a9e4586e51b", "id": "9.Q10a", "kind": "str", "mistakes": {"0e23e85b06f4cd79863179ede8bf3f9fd609cd6b72856984d5e519f5db658344": "un polynôme de degré 15 peut suivre une courbe ondulée, en moyenne sur les jeux"}}
📝 Ex 9.Q10a : réponse enregistrée (str).
WB_ANSWER {"hash": "2eaf7bc01823a4e51152ddd4c564f7155a67d7ff411adfe6a409f5b6f7aae703", "id": "9.Q10b", "kind": "str", "mistakes": {"1a37e187f4d86206b401fa8e3e051d48c162af54e075f45e01de14e4f0be42b3": "des droites ajustées à des jeux de 30 points se ressemblent beaucoup"}}
📝 Ex 9.Q10b : réponse enregistrée (str).
WB_ANSWER {"hash": "c62f0cfea97cccb5173f1b792dfde086e6537beba3da9391cb959fbd4d7e18f6", "id": "9.Q10c", "kind": "bool", "mistakes": {"b7ff3d300c48b1afbd76de88ce44e9647c73758f1ce56af389e71e13ba4df4ee": "où les points qui retiennent le polynôme sont-ils les plus rares ?"}}
📝 Ex 9.Q10c : réponse enregistrée (bool).
WB_ANSWER {"hash": "39d95d3d4aabd9689efd85823f2800

**9.R2 — Ch. 6 : ce que mesure une cross-entropy utilisée comme loss**

In [13]:
wb.record("9.R2a", CE_R2, decimals=3, mistakes={"utilise le logarithme népérien (ln), pas log10": float(-np.mean(np.log10(P_R2))),
          "la cross-entropy est l'opposé du logarithme : elle est positive": -CE_R2,
          "c'est la valeur en bits : la question demande des nats": CE_R2 / math.log(2)})
wb.record("9.R2b", CE_R2 / math.log(2), decimals=3, mistakes={"pour passer des nats aux bits, on divise par ln 2": CE_R2 * math.log(2),
          "c'est la valeur en nats": CE_R2})
wb.record("9.R2c", 3, mistakes={"la plus grande loss vient de la plus petite probabilité donnée à la bonne classe": 1,
          "−ln 0,6 ≈ 0,51 : compare ce terme aux trois autres termes de la somme": 2,
          "−ln 0,8 ≈ 0,22 : compare ce terme aux trois autres termes de la somme": 4})
wb.record("9.R2d", True, mistakes={"une probabilité de 0,9 pour la bonne classe donne-t-elle une loss nulle ?": False})

WB_ANSWER {"decimals": 3, "hash": "f8d4844102fff4cd807eaa50221d13222dcc143db65ed59cc3c87e86f80d23e9", "hash_coarse": "b916ca16e2e6c5665ea75f70b04a2f15315288aba8f421a56b29012516f7e036", "id": "9.R2a", "kind": "float", "magnitude": -1, "mistakes": {"0da8efae609c04fe9535e50777d38c818402b44a94ffa9233ae9797c0584dae6": "la cross-entropy est l'opposé du logarithme : elle est positive", "7ddb812d6f0db8e87686d087b15c990c3b6f5a2ab88587cbca85d20e2fdfa35d": "utilise le logarithme népérien (ln), pas log10", "d98e89ab1c1b6f84bdeea47ef83b1625bbf8d8b087aa860c36267006d6c3c87f": "c'est la valeur en bits : la question demande des nats"}, "sign": 1}
📝 Ex 9.R2a : réponse enregistrée (float, 3 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "68222c9f815f14acfbfffbc98bfc1dee213f83c4813add7eca9787c34a683345", "hash_coarse": "2dcdb1f16b517a08ec8dbba394b065b43d25f3d640fffdc1f5a9405048081946", "id": "9.R2b", "kind": "float", "magnitude": -1, "mistakes": {"6a6ae87b0b85025e9dbeaa556f4b4f909152999d2a7db6c202ac3fcf2

**9.R3 — Ch. 2 : biais et variance d'un estimateur, et le bootstrap**

In [14]:
wb.record("9.R3a", float(S_R3.var()), decimals=3, mistakes={"c'est la division par n − 1 : la question demande la division par n": float(S_R3.var(ddof=1)),
          "c'est la somme des carrés des écarts : divise-la": 26.0})
wb.record("9.R3b", float(S_R3.var(ddof=1)), decimals=3, mistakes={"c'est la division par n : la question demande n − 1": float(S_R3.var())})
wb.record("9.R3c", -0.25, decimals=2, mistakes={"c'est le rapport (n − 1)/n : le biais est la différence avec 1": 0.75,
          "en moyenne, l'estimateur sous-estime la variance : quel est donc le signe du biais ?": 0.25,
          "le biais est E[estimateur] − σ², exprimé en fraction de σ²": -0.75})
wb.record("9.R3d", False, mistakes={"rééchantillonner un échantillon biaisé ne crée pas les personnes qui y manquent": True})

WB_ANSWER {"decimals": 3, "hash": "e58f71eeadee16131e2ee768acb9c1c82b3baaebccb0bd3dbd83d927123df830", "hash_coarse": "d7dea6c8972c122eb70582227fee6d8cf743498ae2c0b554935baad14dbfe646", "id": "9.R3a", "kind": "float", "magnitude": 0, "mistakes": {"b588e9fb1af828f3ba3d97e66cb29beb760aaef672e7405da4177c8f2c16ed51": "c'est la somme des carrés des écarts : divise-la", "d9e2733e76c2f48d06619a52391570221b00ad8e6b17a7852a13053dd6e9bf90": "c'est la division par n − 1 : la question demande la division par n"}, "sign": 1}
📝 Ex 9.R3a : réponse enregistrée (float, 3 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "6847d046c321cb6702c446f8f5a8dcaf80f18b822e7e33fc9b58976908d420d3", "hash_coarse": "b9125fa99aa5e181abbf0bf4d942ec66fa4079d3ca4bbe729e96ffff16d434c1", "id": "9.R3b", "kind": "float", "magnitude": 0, "mistakes": {"1bf2963dc8063d44d00a68de99f517fc25d819011a9851fc64be12e459bf7026": "c'est la division par n : la question demande n − 1"}, "sign": 1}
📝 Ex 9.R3b : réponse enregistrée (float, 3 dé

**Ex 9.1 — MSE et R² à la main sur cinq points**

In [15]:
wb.record("9.1a", float(np.mean(R_91 ** 2)), decimals=3, mistakes={"c'est la somme des carrés : la MSE en est la moyenne": float(np.sum(R_91 ** 2)),
          "on divise par n = 5, pas par n − 1": float(np.sum(R_91 ** 2) / 4)})
wb.record("9.1b", math.sqrt(np.mean(R_91 ** 2)), decimals=3, mistakes={"c'est la MSE : la RMSE est sa racine": float(np.mean(R_91 ** 2))})
wb.record("9.1c", float(np.mean(np.abs(R_91))), decimals=3, mistakes={"les résidus de signes opposés se compensent : prends leurs valeurs absolues": float(np.mean(R_91))})
wb.record("9.1d", 1 - float(np.sum(R_91 ** 2)) / SST_91, decimals=3, mistakes={"c'est le rapport SS_res / SS_tot : R² vaut 1 moins ce rapport": float(np.sum(R_91 ** 2)) / SST_91,
          "SS_tot se calcule autour de la moyenne des cibles": 1 - float(np.sum(R_91 ** 2)) / float(np.sum((Y_91 - YHAT_91.mean()) ** 2))})
wb.record("9.1e", 1 - float(np.sum((Y_91 - 8) ** 2)) / SST_91, decimals=3, mistakes={"un R² peut être négatif : ne l'arrondis pas à 0": 0.0,
          "ce modèle fait moins bien que la moyenne : quel est le signe de son R² ?": -(1 - float(np.sum((Y_91 - 8) ** 2)) / SST_91),
          "c'est le rapport SS_res / SS_tot : R² vaut 1 moins ce rapport": float(np.sum((Y_91 - 8) ** 2)) / SST_91})
wb.record("9.1f", float(np.mean(R2_91 ** 2)), decimals=2, mistakes={"c'est la somme des carrés : divise-la par n": float(np.sum(R2_91 ** 2))})
wb.record("9.1g", float(np.mean(np.abs(R2_91))), decimals=2, mistakes={"c'est la somme des valeurs absolues : divise-la par n": float(np.sum(np.abs(R2_91)))})

WB_ANSWER {"decimals": 3, "hash": "052b1256d0774993c9d62e061910ea41b87958ea906165277a70139b43e78da7", "hash_coarse": "47ad184ad1f0a6acf54d91ea242300fef8d18156ee0c66cbc8afd209487b2790", "id": "9.1a", "kind": "float", "magnitude": -1, "mistakes": {"3ce568a5198fc0922a81e1579748d5e247cdd71b143bb49b077d24aeb57d0b47": "c'est la somme des carrés : la MSE en est la moyenne", "94117e666112774c1d2ebbc9b1e242132535363901b86495cdc7d0cda5341385": "on divise par n = 5, pas par n − 1"}, "sign": 1}
📝 Ex 9.1a : réponse enregistrée (float, 3 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "76bc9541b9e9cc54d315b4aae73f37e550b8848a273f2cc99b6c4aab89e494ed", "hash_coarse": "3f8bc9db56a893cf5fe706efc2245b4f2686da79ee0bbc5f450a7da178e5c7fe", "id": "9.1b", "kind": "float", "magnitude": -1, "mistakes": {"f9f51b85e227781d3f99196dc829a9ebd9d2409545b56a3245f8acfcb8762c5f": "c'est la MSE : la RMSE est sa racine"}, "sign": 1}
📝 Ex 9.1b : réponse enregistrée (float, 3 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": 

**Ex 9.2 — Moindres carrés : la meilleure droite par dérivées partielles**

In [16]:
wb.record("9.2a", A_92, decimals=3, mistakes={"c'est la pente de la droite forcée de passer par l'origine : centre les données": float(X_92 @ Y_92 / (X_92 @ X_92))})
wb.record("9.2b", B_92, decimals=3, mistakes={"l'ordonnée à l'origine vaut ȳ − a·x̄": float(Y_92.mean() - A_92)})
wb.record("9.2c", round(float(np.sum(RES_92))), fractional="la démonstration 3 donne cette somme sans calcul : vérifie-la avec la pente et l'ordonnée de a) et b), en gardant leurs valeurs exactes")
wb.record("9.2d", float(np.sum(RES_92 ** 2)), decimals=3, mistakes={"c'est la MSE : la question demande la somme des carrés": float(np.mean(RES_92 ** 2))})
wb.record("9.2e", A_92 * 6 + B_92, decimals=3, mistakes={"la prédiction est a·x + b": A_92 * 6})
wb.record("9.2f", 5 * 30 - 10 ** 2, mistakes={"le déterminant de [[p, q], [q, r]] vaut p·r − q²": 5 * 30})

WB_ANSWER {"decimals": 3, "hash": "72c5da3b10d64c4d1bb3f68606ca38301af1b71f317c6f085ec3103f51a6ae2a", "hash_coarse": "150e1630549f317863163a4bec41a3ee781ca17e1194480e32b22e4232464f6c", "id": "9.2a", "kind": "float", "magnitude": 0, "mistakes": {"4859e0087b38178db93751ae34761a4a7badb0f61e4ef32c71ce408df3720ce5": "c'est la pente de la droite forcée de passer par l'origine : centre les données"}, "sign": 1}
📝 Ex 9.2a : réponse enregistrée (float, 3 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "b77e1c382c11b49476dc92bfd712fb3abff2cf8bacc4494fd7cc75e4477f4505", "hash_coarse": "3cad4ff0e8862b3f5212ba6564297d631a0fd6343d8b2b0bbd9c0856dd522c86", "id": "9.2b", "kind": "float", "magnitude": -1, "mistakes": {"cc80a501590623d62d003da7452d2f740fe49d9208a0cc3bcc7ba0f815c1c47b": "l'ordonnée à l'origine vaut ȳ − a·x̄"}, "sign": 1}
📝 Ex 9.2b : réponse enregistrée (float, 3 décimale(s)).
WB_ANSWER {"fractional": "la démonstration 3 donne cette somme sans calcul : vérifie-la avec la pente et l'ordonné

**Ex 9.3 — Ridge en dimension 1 : w* = Σxy / (Σx² + λ)**

In [17]:
wb.record("9.3a", ridge_1d(0), decimals=3)
wb.record("9.3b", ridge_1d(2.5), decimals=3, mistakes={"λ s'ajoute à Σx², au dénominateur, pas au poids": ridge_1d(0) + 2.5})
wb.record("9.3c", ridge_1d(5), decimals=4, mistakes={"λ s'ajoute à Σx², au dénominateur": SXY_93 / (SXX_93 * 5)})
wb.record("9.3d", SXX_93, decimals=3, mistakes={"c'est la valeur de Σxy": SXY_93,
          "c'est la valeur du dénominateur Σx² + λ : la question demande λ": 2 * SXX_93})
wb.record("9.3e", float(np.sum((Y_93 - ridge_1d(10) * X_93) ** 2)), decimals=3, mistakes={"la question demande la somme des carrés des résidus seule, sans ajouter la pénalité": float(np.sum((Y_93 - ridge_1d(10) * X_93) ** 2)) + 10 * ridge_1d(10) ** 2,
          "c'est la somme des carrés sans pénalité (λ = 0)": float(np.sum((Y_93 - ridge_1d(0) * X_93) ** 2))})
wb.record("9.3f", True, mistakes={"la somme des carrés est minimale au poids des moindres carrés : que se passe-t-il quand w* s'en éloigne ?": False})

WB_ANSWER {"decimals": 3, "hash": "d2189ed5d1ca2cb0df783cf2e1d8583672529b7a88a1ecdf2d77b893b899a02b", "hash_coarse": "2aaf569444ec331ebaa14358c14adb765b56181e280e36f16b560f0ebbb85aab", "id": "9.3a", "kind": "float", "magnitude": 0, "sign": 1}
📝 Ex 9.3a : réponse enregistrée (float, 3 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "0dc9dbc5b10a390e7d59011e30d3dfe518595cd726b0e359542336d926e594a6", "hash_coarse": "ac8d77b5ecdb9b17b82218b805f4a2d0ae963236d23eb9efb889caed6c241639", "id": "9.3b", "kind": "float", "magnitude": 0, "mistakes": {"74ec57f748eae17e9232ce7f4caee839e485cbc7e4824431ddb12743c14eb5b4": "λ s'ajoute à Σx², au dénominateur, pas au poids"}, "sign": 1}
📝 Ex 9.3b : réponse enregistrée (float, 3 décimale(s)).
WB_ANSWER {"decimals": 4, "hash": "6254b7b4ce27953bc5d491a6aa1a6dc3229a63efca376f327201481eb096f29e", "hash_coarse": "029c9f2ba31b18c5aa852a08a712183afab45e6472f70f32f5b57e0bbcdc942b", "id": "9.3c", "kind": "float", "magnitude": -1, "mistakes": {"18e5c7dc8f3de54d5aa271

**Ex 9.4 — Biais² et variance à partir d'un tableau de prédictions**

In [18]:
wb.record("9.4a", AVG_94.tolist(), decimals=3, mistakes={"ce sont les valeurs de la courbe idéale f ; le modèle moyen se calcule à partir des quatre modèles": F_94.tolist()})
wb.record("9.4b", BIAS2_94, decimals=3, mistakes={"élève chaque écart au carré avant de faire la moyenne": float(np.mean(AVG_94 - F_94) ** 2),
          "c'est l'erreur quadratique moyenne des modèles ; le biais² se calcule avec le modèle moyen": float(np.mean((P_94 - F_94) ** 2)),
          "le biais² est une moyenne de carrés : il ne peut pas être négatif": float(np.mean(AVG_94 - F_94))})
wb.record("9.4c", VAR_94, decimals=3, mistakes={"la variance divise par le nombre de modèles (4), comme bias_variance_decomposition": float(np.mean(P_94.var(axis=0, ddof=1)))})
wb.record("9.4d", float(np.mean((P_94 - F_94) ** 2)), decimals=3, mistakes={"c'est le biais² seul : la question porte sur les douze erreurs des modèles": BIAS2_94,
          "divise la somme par le nombre de couples (modèle, point)": float(np.sum((P_94 - F_94) ** 2) / 4)})
wb.record("9.4e", float(np.mean((P_94 - F_94) ** 2)) + 0.09, decimals=3, mistakes={"sur des mesures bruitées, il faut ajouter le bruit": float(np.mean((P_94 - F_94) ** 2)),
          "le bruit s'ajoute par sa variance σ², pas par son écart-type": float(np.mean((P_94 - F_94) ** 2)) + 0.3})
wb.record("9.4f", True, mistakes={"l'erreur du modèle moyen face à f, c'est l'un des termes déjà calculés : compare-le à d)": False})

WB_ANSWER {"decimals": 3, "element_coarse": [["47643de286d917e11a66d4f9e55a36b9864bbd5633445c4ea01e2e81ca737202"], ["5741c16933515af952bbf60799db420cb6d3c04390206c81e8e74107e955190c"], ["430d1dc0955617a367e32cfe5b21360dbc3b0bbe81da6a254f347501566622e7"]], "element_hashes": ["4b402183f8260dd6f205a7dbbdcd281efd5fa5160f32a1e59296994135dc9ffd", "9783d11748db330b7c4da85c9ae4911445d8cd789c2fb9b03852656c75ff10d5", "302ab48dfe07bcb0c001a9d64a0ae7e64a040d6f325657b7666a3d3d53619ea1"], "hash": "10f6047987904e20323ebae64b073d0a23be385a208c0492ab0d50bfcf5d8b1e", "id": "9.4a", "kind": "array", "mistakes": {"6fb52e8e72991fc880ddd0901f6e6c383897f4479e63d48eff61c820b1ccf1cc": "ce sont les valeurs de la courbe idéale f ; le modèle moyen se calcule à partir des quatre modèles"}, "shape": [3]}
📝 Ex 9.4a : réponse enregistrée (array, 3 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "859cbc6cf87cc5c12f90fe08a9fdd2edd8105a6cf6bb3384261bdc26efcb2fde", "hash_coarse": "4e54e061057dd2ee45c6393ddd707c11cba0de9e4

**Ex 9.5 — Early stopping avec patience sur une courbe de loss**

In [19]:
wb.record("9.5a", early_stopping(VAL_95, 2)[0], mistakes={"l'epoch de la meilleure loss ne compte pas dans l'attente": 8,
          "une loss de 0,52 est une amélioration sur 0,53 : l'attente repart de zéro": 7})
wb.record("9.5b", early_stopping(VAL_95, 2)[1], mistakes={"on recharge les poids de la meilleure epoch, pas ceux de l'arrêt": 9,
          "une loss de 0,52 bat la meilleure loss précédente": 5})
wb.record("9.5c", early_stopping(VAL_95, 1)[0], mistakes={"avec une patience de 1, une seule epoch sans amélioration suffit à arrêter": 9,
          "on s'arrête à la fin d'une epoch sans amélioration, pas à la fin de la meilleure": 5})
wb.record("9.5d", early_stopping(VAL_95, 1)[1], mistakes={"on recharge les poids de la meilleure epoch, pas ceux de l'arrêt": 6,
          "cette epoch n'a pas été atteinte : l'entraînement s'est arrêté avant": 7})
wb.record("9.5e", early_stopping(VAL_95, 4)[0], mistakes={"l'epoch de la meilleure loss ne compte pas dans l'attente": 10,
          "une loss de 0,52 est une amélioration sur 0,53 : l'attente repart de zéro": 9})
wb.record("9.5f", early_stopping(VAL_95, 2, 0.02)[0], mistakes={"avec min_delta, une baisse ne compte que si elle dépasse 0,02 : refais le suivi": 9,
          "une baisse d'exactement 0,02 n'est pas une amélioration avec min_delta = 0,02 (comparaison stricte)": 7})
wb.record("9.5g", early_stopping(VAL_95, 2, 0.02)[1], mistakes={"une baisse d'exactement 0,02 n'est pas une amélioration avec min_delta = 0,02 (comparaison stricte)": 5,
          "avec min_delta, une baisse ne compte que si elle dépasse 0,02 : refais le suivi": 7,
          "on recharge les poids de la dernière amélioration, pas ceux de l'epoch d'arrêt": 6})

WB_ANSWER {"hash": "aaee722ec9a3d4110a3b4325f616413ff9a7aea2ad3c18a12e743d88b1a1f65c", "id": "9.5a", "kind": "int", "magnitude": 0, "mistakes": {"565106c73bacd5587499e2c56047c772080c0fec4f8422880f7adad1176e2551": "l'epoch de la meilleure loss ne compte pas dans l'attente", "d5de3336fa34e80bfff6201f197c08f253c777e3dfbc3305207a3761453da2c9": "une loss de 0,52 est une amélioration sur 0,53 : l'attente repart de zéro"}, "sign": 1}
📝 Ex 9.5a : réponse enregistrée (int).
WB_ANSWER {"hash": "29ac1a6fa213c2bf79196507a5ef0c270642f109405953b318a77166416ec997", "id": "9.5b", "kind": "int", "magnitude": 0, "mistakes": {"ca71619d9606010296a1d000d0e08d1811be034fa54058124617f1d7523989ff": "une loss de 0,52 bat la meilleure loss précédente", "edc6f616637dde7af18cb3ab812e94f1acef66d3297d429bb27bf437cc35ab56": "on recharge les poids de la meilleure epoch, pas ceux de l'arrêt"}, "sign": 1}
📝 Ex 9.5b : réponse enregistrée (int).
WB_ANSWER {"hash": "f20cd9600bff711fc8553d29b6cdb484a86cf623ef35aaca9e0a3cd1d

**Ex 9.6 — Lasso en dimension 1 : le seuillage doux et les zéros exacts**

In [20]:
wb.record("9.6a", soft_threshold(2.5, 1), decimals=3, mistakes={"le seuillage rapproche z de 0 : pour un z positif, on retire γ": 3.5})
wb.record("9.6b", soft_threshold(-0.4, 0.5), decimals=3, mistakes={"si |z| ≤ γ, le seuillage doux donne exactement 0": -0.4 + 0.5,
          "c'est z sans seuillage : compare d'abord |z| au seuil γ": -0.4,
          "le seuillage rapproche z de 0, il ne l'en éloigne jamais": -0.4 - 0.5})
wb.record("9.6c", soft_threshold(-3, 0.5), decimals=3, mistakes={"le seuillage rapproche z de 0 : pour un z négatif, on ajoute γ": -3.5,
          "le seuillage doux garde le signe de z": 2.5})
wb.record("9.6d", RHO_96, decimals=3, mistakes={"ρ divise Σxy par n": SXY_93,
          "c'est le poids des moindres carrés, Σxy / Σx²": SXY_93 / SXX_93})
wb.record("9.6e", soft_threshold(RHO_96, 0.6) / Z_96, decimals=3, mistakes={"le seuillage doux se divise ensuite par z = Σx² / n": soft_threshold(RHO_96, 0.6),
          "le seuil porte sur ρ, avant la division par z": RHO_96 / Z_96 - 0.6})
wb.record("9.6f", soft_threshold(RHO_96, 1.3) / Z_96, decimals=3, mistakes={"le seuillage doux se divise ensuite par z = Σx² / n": soft_threshold(RHO_96, 1.3)})
wb.record("9.6g", abs(RHO_96), decimals=3, mistakes={"c'est le poids des moindres carrés : le seuil porte sur ρ": SXY_93 / SXX_93})
wb.record("9.6h", True, mistakes={"relis la démonstration 2 de l'exercice 9.3 : le numérateur de w* change-t-il avec λ ?": False})

WB_ANSWER {"decimals": 3, "hash": "424f7f58b701b13d6d48aa3f019192597780a85da0445f3679ff87c650d17488", "hash_coarse": "fd078f477226183ae3b0cfbc7ab3cdaf18145214629a6c12d117c8a5c4d2ad12", "id": "9.6a", "kind": "float", "magnitude": 0, "mistakes": {"564a4eefadf9b2425708121372ea0794f0efcb92afbe666f1df1c87c9292dd1f": "le seuillage rapproche z de 0 : pour un z positif, on retire γ"}, "sign": 1}
📝 Ex 9.6a : réponse enregistrée (float, 3 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "51ed8dae759b1e17a78622036a8d2be1e86f8bfae94c48bf77a8a1a2f09c9467", "hash_coarse": "78352c2f2ed9414871d012146a92f32162e5d427e2afe4092dc3a1584e3da27e", "id": "9.6b", "kind": "float", "magnitude": null, "mistakes": {"09275a01bc0d386f75f53f7baf4539cc4708ac3cc3f96290775e795450615415": "si |z| ≤ γ, le seuillage doux donne exactement 0", "3986f7b7bbb42dcd0f3c1c3ff9a82bb971c5c2b3c3acef213b85eb1c53737ab1": "le seuillage rapproche z de 0, il ne l'en éloigne jamais", "7c097f97d8ae69b66b116eac2f50434771f681eea0ec6675839e47a3

**Ex 9.7 — Mise à jour bayésienne d'une droite sur une grille 3 × 3**

In [21]:
wb.record("9.7a", PRIOR_97[(0, 0)], decimals=3, mistakes={"c'est un poids : divise-le par la somme des neuf poids": 4})
wb.record("9.7b", POST1_97[(0, 1)], decimals=3, mistakes={"multiplie le prior par la vraisemblance, puis normalise par la somme des neuf produits": PRIOR_97[(0, 1)] * likelihood_97((0, 1), (1, 1)),
          "un écart de 3 a une vraisemblance nulle, pas 0,1 : refais la somme des neuf produits": 0.25})
wb.record("9.7c", sum(1 for line in LINES_97 if POST1_97[line] == 0), mistakes={"une probabilité a posteriori est nulle quand la vraisemblance est nulle : regarde les écarts de 3 ou plus": 0})
wb.record("9.7d", POST2_97[(0, 0)], decimals=3, mistakes={"multiplie par la vraisemblance du second point, puis normalise par la somme des neuf nouveaux produits": POST1_97[(0, 0)] * likelihood_97((0, 0), (-1, 0))})
wb.record("9.7e", POST2_97[(1, 1)], decimals=3, mistakes={"c'est la probabilité après le premier point seulement": POST1_97[(1, 1)]})
wb.record("9.7f", True, mistakes={"le posterior final est proportionnel au produit prior × L1 × L2 : l'ordre des facteurs compte-t-il ?": False})
wb.record("9.7g", list(max(LINES_97, key=POST2_97.get)), decimals=1, mistakes={"c'est la droite exacte de h), mais elle n'est pas dans la grille : la réponse est l'une des neuf droites": [0.5, 0.5],
          "c'est l'une des droites les plus probables après P1 seulement : tiens compte aussi de P2": [0, 1],
          "après P1 seulement, cette droite était à égalité en tête : multiplie aussi par la vraisemblance de P2": [1, 0],
          "elle passe exactement par P2, mais pas par P1, et son prior est faible : compare les produits prior × L1 × L2": [1, 1]})

WB_ANSWER {"decimals": 3, "hash": "2ce56536106e1551039f0c9f6877eb54d7b805f988ce0be0fee27067f47f6971", "hash_coarse": "71c2a687c35177fd12438d8a592c608d53a7388b9bbd0575df460bf0d69cc481", "id": "9.7a", "kind": "float", "magnitude": -1, "mistakes": {"607e7b79e9db528da25d788027d0f2d0c1dc06d19164454cc3c5a21573b20078": "c'est un poids : divise-le par la somme des neuf poids"}, "sign": 1}
📝 Ex 9.7a : réponse enregistrée (float, 3 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "8b1be946750254f4f568f5ee2bdd4bf26914b2fe7480c1db896ae472c76707e8", "hash_coarse": "18c6783bb96310ce2ceb6038bb56493b0adeb68885721c686a321c9d00029f3d", "id": "9.7b", "kind": "float", "magnitude": -1, "mistakes": {"77e48f07fd432a906ef1792483e395c307e8bf95bd0c111735b4af4eb617172c": "un écart de 3 a une vraisemblance nulle, pas 0,1 : refais la somme des neuf produits", "eb129a50d665b3d1f64b357571a3d22b3f0df83ccc3bda58ee6e9aef1d906b85": "multiplie le prior par la vraisemblance, puis normalise par la somme des neuf produits"},

**Ex 9.9 — Diagnostiquer quatre paires de courbes d'entraînement et de validation**

In [22]:
wb.record("9.9a", 4, mistakes={"dans ce panneau, la loss de validation reste sous celle d'entraînement : ce n'est pas le symptôme cherché": 2,
          "dans ce panneau, la loss d'entraînement est très basse : le modèle réussit ses propres exemples": 1,
          "dans ce panneau, les deux losses descendent bas et restent proches : le modèle apprend sans rester bloqué": 3})
wb.record("9.9b", 1, mistakes={"dans ce panneau, les deux losses sont hautes et collées": 4,
          "dans ce panneau, les deux losses descendent bas et restent proches": 3,
          "dans ce panneau, la loss de validation reste sous celle d'entraînement : ce n'est pas le symptôme cherché": 2})
wb.record("9.9c", 30, choices=[10, 30, 60, 100],
          mistakes={"c'est la fin de l'entraînement : compare la loss de validation à cet endroit avec celle des autres choix": 100,
          "lis la loss de validation à cette epoch et compare-la avec celle des autres choix": 60,
          "suis la courbe de validation (orange) sur toute sa longueur, et cherche son point le plus bas": 10})
wb.record("9.9d", "AC", mistakes={"un modèle qui sous-apprend a une validation au-dessus de l'entraînement, pas au-dessous": "ABC",
          "un learning rate trop petit ralentit les deux courbes sans changer leur ordre": "ACD",
          "une validation plus facile que l'entraînement (cas plus nets, labels plus sûrs) explique aussi ce panneau": "A",
          "une technique active seulement pendant l'entraînement explique aussi ce panneau": "C"})
wb.record("9.9e", False, mistakes={"le panneau 4 montre de l'underfitting : plus de données le rendrait-il plus souple ?": True})

WB_ANSWER {"hash": "51c0931baebf33446e4d1ebf9e0c5f67bb9d955ecf3643b89321c90892186626", "id": "9.9a", "kind": "int", "magnitude": 0, "mistakes": {"2c529d86566f44a6ff7ccc525c2de2a6cf702e215fa5644f4f7544923a9c7e95": "dans ce panneau, la loss de validation reste sous celle d'entraînement : ce n'est pas le symptôme cherché", "5bf4412f4734422217aca57aefb74429d76963178c599f46f899cdae616d172d": "dans ce panneau, les deux losses descendent bas et restent proches : le modèle apprend sans rester bloqué", "75990814476db3a161347a6ffbb8c7061cc29a4527adc80cf3aa44779eda8b9c": "dans ce panneau, la loss d'entraînement est très basse : le modèle réussit ses propres exemples"}, "sign": 1}
📝 Ex 9.9a : réponse enregistrée (int).
WB_ANSWER {"hash": "32ae19037ae5aa2337c13e66d2b8ad8facc349c8b38cee139a44a77da41337ee", "id": "9.9b", "kind": "int", "magnitude": 0, "mistakes": {"30303d267983664a40b59c7c9f8823888b90b23da83c53fcc5bb0b6d671973b0": "dans ce panneau, la loss de validation reste sous celle d'entraînemen

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu dire, à partir de deux erreurs et d'une référence, si un modèle sous-apprend ou sur-apprend, et quoi faire dans chaque cas ?
2. Sais-tu dériver les moindres carrés et Ridge en dimension 1, et expliquer pourquoi le Lasso met des poids exactement à zéro ?
3. Sais-tu calculer le biais² et la variance d'une famille de modèles, et dire pourquoi le compromis biais-variance n'est pas une loi ?

**Pour aller plus loin** : le guide « Linear Models » de scikit-learn et le chapitre 6 d'*An Introduction to Statistical Learning*, cités dans la fiche. La suite : le ch. 10 (neurones), où un modèle linéaire suivi d'une activation devient un neurone.

*Les exercices de code 9.12 à 9.31 (parties A à D, ta librairie `mylearn.linear`) seront ajoutés à la prochaine session de génération ; relance alors `python tools/start_chapter.py 9` pour obtenir le notebook complet.*